# 05 CV Structured Extraction

This notebook extracts structured information from a CV text.

Main goals:
- load extracted CV text from the previous PDF extraction step
- use OpenAI `gpt-4o-mini` through LangChain
- define a structured Pydantic schema for CV data
- extract candidate profile information, skills, tools, experience, education, projects and certifications
- save the structured CV data as JSON

This step does not evaluate CV quality and does not compare the CV with a job posting.

## 1. Imports and Settings

In [1]:
import os
import json
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from typing import List, Optional
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [2]:
load_dotenv(dotenv_path=Path("../.env"))
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

In [3]:
PROCESSED_TEST_CV_DIR = Path("../data/processed/test_cv")
OUTPUT_CV_EXTRACTION_DIR = Path("../outputs/cv_extraction")

cv_text_path = PROCESSED_TEST_CV_DIR / "test_cv_extracted_text.txt"
structured_cv_output_path = OUTPUT_CV_EXTRACTION_DIR / "structured_cv.json"

In [4]:
cv_text_path

WindowsPath('../data/processed/test_cv/test_cv_extracted_text.txt')

## 2. Load Extracted CV Text

In [5]:
with open(cv_text_path, "r", encoding="utf-8") as file:
    cv_text=file.read()

## 3. Basic Text Check

In [6]:
cv_text_stats={
    "character_count": len(cv_text),
    "word_count":len(cv_text.split()),
    "line_count": len(cv_text.splitlines())
}

In [7]:
cv_text_stats

{'character_count': 2443, 'word_count': 327, 'line_count': 45}

## 4. Define Structured CV Schema

In [8]:
class EducationItem(BaseModel):
    institution: Optional[str]=Field(
        default=None,
        decription="Name of the school, university or educational institution."
    )
    degree: Optional[str]= Field(
        default=None,
        description="Degree or qualification, if visible in the CV."
    )
    field_of_study: Optional[str] = Field(
        default=None,
        description="Field of study, if visible in the CV."
    )
    start_year: Optional[str] = Field(
        default=None,
        description="Start year or date, if visible."
    )
    end_year: Optional[str] = Field(
        default=None,
        description="End year or date, if visible."
    )

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7920\2413388717.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'decription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  institution: Optional[str]=Field(


In [9]:
class ExperienceItem(BaseModel):

    company: Optional[str] = Field(
        default=None,
        description="Company or organization name, if visible."
    )
    position: Optional[str] = Field(
        default=None,
        description="Job title or role."
    )
    start_date: Optional[str] = Field(
        default=None,
        description="Start date, if visible."
    )
    end_date: Optional[str] = Field(
        default=None,
        description="End date, if visible. Use Present if the CV clearly says the role is current."
    )
    responsibilities: List[str] = Field(
        default_factory=list,
        description="Main responsibilities and tasks explicitly mentioned in the CV."
    )
    technologies_used: List[str] = Field(
        default_factory=list,
        description="Technologies, tools or programming languages explicitly connected to this experience."
    )

In [10]:
class ProjectItem(BaseModel):

    project_name: Optional[str] = Field(
        default=None,
        description="Project name, if visible."
    )
    description: Optional[str] = Field(
        default=None,
        description="Short description of the project."
    )
    technologies_used: List[str] = Field(
        default_factory=list,
        description="Technologies, tools or programming languages explicitly mentioned for this project."
    )
    project_result: Optional[str] = Field(
        default=None,
        description="Result or outcome of the project, if visible."
    )

In [11]:
class CertificationItem(BaseModel):

    name: Optional[str] = Field(
        default=None,
        description="Certification name."
    )
    issuer: Optional[str] = Field(
        default=None,
        description="Certification issuer or organization, if visible."
    )
    year: Optional[str] = Field(
        default=None,
        description="Year or date of certification, if visible."
    )

In [12]:
class StructuredCV(BaseModel):
    candidate_name: Optional[str] = Field(
        default=None,
        description="Candidate full name, if visible in the CV."
    )

    email: Optional[str] = Field(
        default=None,
        description="Candidate email address, if visible."
    )

    phone: Optional[str] = Field(
        default=None,
        description="Candidate phone number, if visible."
    )

    location: Optional[str] = Field(
        default=None,
        description="Candidate location, if visible."
    )

    linkedin_url: Optional[str] = Field(
        default=None,
        description="LinkedIn profile URL, if visible."
    )

    github_url: Optional[str] = Field(
        default=None,
        description="GitHub profile URL, if visible."
    )

    portfolio_url: Optional[str] = Field(
        default=None,
        description="Portfolio or personal website URL, if visible."
    )

    profile_summary: Optional[str] = Field(
        default=None,
        description="Short professional summary based only on the CV text."
    )

    total_years_of_experience: Optional[str] = Field(
        default=None,
        description="Total years of experience only if clearly visible or directly inferable from dates."
    )

    technical_skills: List[str] = Field(
        default_factory=list,
        description="Technical skills explicitly mentioned in the CV."
    )

    programming_languages: List[str] = Field(
        default_factory=list,
        description="Programming languages explicitly mentioned in the CV."
    )

    frameworks_and_libraries: List[str] = Field(
        default_factory=list,
        description="Frameworks and libraries explicitly mentioned in the CV."
    )

    databases: List[str] = Field(
        default_factory=list,
        description="Databases explicitly mentioned in the CV."
    )

    cloud_and_devops_tools: List[str] = Field(
        default_factory=list,
        description="Cloud platforms, DevOps tools and infrastructure tools explicitly mentioned in the CV."
    )

    data_and_ai_tools: List[str] = Field(
        default_factory=list,
        description="Data analysis, machine learning, AI or BI tools explicitly mentioned in the CV."
    )

    other_tools: List[str] = Field(
        default_factory=list,
        description="Other software tools explicitly mentioned in the CV."
    )

    soft_skills: List[str] = Field(
        default_factory=list,
        description="Soft skills explicitly mentioned in the CV."
    )

    languages: List[str] = Field(
        default_factory=list,
        description="Human languages explicitly mentioned in the CV."
    )

    education: List[EducationItem] = Field(
        default_factory=list,
        description="Education entries extracted from the CV."
    )

    work_experience: List[ExperienceItem] = Field(
        default_factory=list,
        description="Work experience entries extracted from the CV."
    )

    projects: List[ProjectItem] = Field(
        default_factory=list,
        description="Project entries extracted from the CV."
    )

    certifications: List[CertificationItem] = Field(
        default_factory=list,
        description="Certifications explicitly mentioned in the CV."
    )

    unclear_or_missing_information: List[str] = Field(
        default_factory=list,
        description="Important information that is missing or unclear in the CV."
    )

## 5. Initialize OpenAI Model

In [13]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [14]:
structured_llm = llm.with_structured_output(StructuredCV)

## 6. Create CV Structured Extraction Prompt

In [15]:
cv_extraction_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI assistant specialized in structured CV extraction for IT candidates.

Your task is to extract structured information from the provided CV text.

Important rules:
- Extract only information that is explicitly present in the CV text.
- Do not invent skills, tools, programming languages, certifications, education, projects or work experience.
- Do not assume that the candidate has a skill if it is not visible in the CV.
- If information is missing or unclear, leave the field empty or add a note to unclear_or_missing_information.
- Keep extracted skill names concise and normalized when possible.
- Focus on IT-related information such as programming languages, frameworks, databases, cloud tools, DevOps tools, data tools, AI tools and software tools.
- Do not evaluate the CV quality in this step.
- Do not compare the CV with a job posting in this step.

Return the result using the required structured schema.
"""
        ),
        (
            "human",
            """
Extract structured information from the following CV text:

{cv_text}
"""
        )
    ]
)

## 7. Run Structured CV Extraction

In [16]:
cv_extraction_chain=cv_extraction_prompt | structured_llm

In [17]:
structured_cv_result=cv_extraction_chain.invoke(
    {
        "cv_text":cv_text
    }
)

In [18]:
structured_cv_result

StructuredCV(candidate_name='ALEX JOHNSON', email='alex.jjohnson@example.com', phone='+1 202-555-0147', location='Example City, 10000', linkedin_url='https://www.linkedin.com/in/alex-johnson-example/', github_url='', portfolio_url='', profile_summary='Highly motivated Third-Year Software Engineering student (GPA 9.2/10.0) at the School of Electrical Engineering (ETF), University of Belgrade, specializing in Data Structures and Algorithms (DSA) and System Programming (C/C++, Java, Python). Seeking to leverage advanced technical skills and a robust background in low-level systems and application development to contribute high-impact solutions to a challenging and innovative engineering environment. Strong communication and teamwork skills developed through university projects.', total_years_of_experience='', technical_skills=['Algorithms', 'Data Structures', 'Problem solving', 'Networks'], programming_languages=['C', 'C++', 'Java', 'Python', 'SQL', 'Assembly'], frameworks_and_libraries=[

## 8. Convert Structured Result to Dictionary

In [19]:
if hasattr(structured_cv_result, "model_dump"):
    structured_cv_dict = structured_cv_result.model_dump()
else:
    structured_cv_dict = structured_cv_result.dict()

structured_cv_dict

{'candidate_name': 'ALEX JOHNSON',
 'email': 'alex.jjohnson@example.com',
 'phone': '+1 202-555-0147',
 'location': 'Example City, 10000',
 'linkedin_url': 'https://www.linkedin.com/in/alex-johnson-example/',
 'github_url': '',
 'portfolio_url': '',
 'profile_summary': 'Highly motivated Third-Year Software Engineering student (GPA 9.2/10.0) at the School of Electrical Engineering (ETF), University of Belgrade, specializing in Data Structures and Algorithms (DSA) and System Programming (C/C++, Java, Python). Seeking to leverage advanced technical skills and a robust background in low-level systems and application development to contribute high-impact solutions to a challenging and innovative engineering environment. Strong communication and teamwork skills developed through university projects.',
 'total_years_of_experience': '',
 'technical_skills': ['Algorithms',
  'Data Structures',
  'Problem solving',
  'Networks'],
 'programming_languages': ['C', 'C++', 'Java', 'Python', 'SQL', 'A

## 9. Preview Extracted Candidate

In [20]:
candidate_profile = {
    "candidate_name": structured_cv_dict.get("candidate_name"),
    "email": structured_cv_dict.get("email"),
    "phone": structured_cv_dict.get("phone"),
    "location": structured_cv_dict.get("location"),
    "linkedin_url": structured_cv_dict.get("linkedin_url"),
    "github_url": structured_cv_dict.get("github_url"),
    "portfolio_url": structured_cv_dict.get("portfolio_url"),
    "total_years_of_experience": structured_cv_dict.get("total_years_of_experience")
}

In [21]:
candidate_profile

{'candidate_name': 'ALEX JOHNSON',
 'email': 'alex.jjohnson@example.com',
 'phone': '+1 202-555-0147',
 'location': 'Example City, 10000',
 'linkedin_url': 'https://www.linkedin.com/in/alex-johnson-example/',
 'github_url': '',
 'portfolio_url': '',
 'total_years_of_experience': ''}

In [22]:
skills_preview = {
    "technical_skills": structured_cv_dict.get("technical_skills", []),
    "programming_languages": structured_cv_dict.get("programming_languages", []),
    "frameworks_and_libraries": structured_cv_dict.get("frameworks_and_libraries", []),
    "databases": structured_cv_dict.get("databases", []),
    "cloud_and_devops_tools": structured_cv_dict.get("cloud_and_devops_tools", []),
    "data_and_ai_tools": structured_cv_dict.get("data_and_ai_tools", []),
    "other_tools": structured_cv_dict.get("other_tools", [])
}

In [23]:
skills_preview

{'technical_skills': ['Algorithms',
  'Data Structures',
  'Problem solving',
  'Networks'],
 'programming_languages': ['C', 'C++', 'Java', 'Python', 'SQL', 'Assembly'],
 'frameworks_and_libraries': [],
 'databases': [],
 'cloud_and_devops_tools': [],
 'data_and_ai_tools': [],
 'other_tools': ['Git']}

## 10. Convert Skill Lists to DataFrame

In [24]:
skill_rows = []

skill_categories = [
    "technical_skills",
    "programming_languages",
    "frameworks_and_libraries",
    "databases",
    "cloud_and_devops_tools",
    "data_and_ai_tools",
    "other_tools",
    "soft_skills",
    "languages"
]

for category in skill_categories:
    for skill in structured_cv_dict.get(category, []):
        skill_rows.append(
            {
                "category": category,
                "skill": skill
            }
        )

skills_df = pd.DataFrame(skill_rows)

In [25]:
skills_df

,category,skill
0,technical_skills,Algorithms
1,technical_skills,Data Structures
2,technical_skills,Problem solving
3,technical_skills,Networks
4,programming_languages,C
5,programming_languages,C++
6,programming_languages,Java
7,programming_languages,Python
8,programming_languages,SQL
9,programming_languages,Assembly


In [26]:
work_experience_df = pd.DataFrame(structured_cv_dict.get("work_experience", []))

In [27]:
work_experience_df

""


In [28]:
projects_df = pd.DataFrame(structured_cv_dict.get("projects", []))

In [29]:
projects_df

,project_name,description,technologies_used,project_result
0,Multithreaded Preemptive Kernel for RISC-V Arc...,Developed a foundational kernel model to demon...,"[C++, C, Assembly]",
1,Command-Line Interpreter,Developed a functional command-line interprete...,[C++],
2,GUI Flight Simulator,Developed a visual flight simulator using Java...,[Java],
3,Sudoku & Ludo,"Developed interactive, fully functional game a...",[Python],
4,"Binary Tree, Fibonacci heap, B* Tree and Graph...",Implemented performance-critical algorithms fo...,"[C, C++]",


In [30]:
education_df = pd.DataFrame(structured_cv_dict.get("education", []))

In [31]:
education_df

,institution,degree,field_of_study,start_year,end_year
0,University of Belgrade - School of Electrical ...,Bachelor of Science (B.Sc.),Software Engineering,,June 2027


In [32]:
certifications_df = pd.DataFrame(structured_cv_dict.get("certifications", []))

In [33]:
certifications_df

""


## 15. Add Metadata

In [34]:
structured_cv_dict["metadata"] = {
    "source_file": cv_text_path.name,
    "model": "gpt-4o-mini",
    "extraction_type": "structured_cv_extraction",
    "notes": "Only information visible in the CV text should be included."
}

In [35]:
structured_cv_dict["metadata"]

{'source_file': 'test_cv_extracted_text.txt',
 'model': 'gpt-4o-mini',
 'extraction_type': 'structured_cv_extraction',
 'notes': 'Only information visible in the CV text should be included.'}

## 16. Save Structured CV JSON

In [36]:
with open(structured_cv_output_path, "w", encoding="utf-8") as file:
    json.dump(structured_cv_dict, file, indent=4, ensure_ascii=False)

In [37]:
print(f"Structured CV JSON saved to: {structured_cv_output_path}")

Structured CV JSON saved to: ..\outputs\cv_extraction\structured_cv.json
